# SFH14 Load Forecast Notebook

This notebook isolates `load` training, diagnostics, and offline repair experiments for `SFH14` while keeping `SFH12` and `SFH16` as guardrail controls.


## 1. Environment and Paths

Bootstrap the project root, imports, and display helpers before editing any experiment settings.


In [ ]:
from pathlib import Path
import sys
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "configs").exists():
    project_root = project_root.parent
if not (project_root / "configs").exists():
    raise RuntimeError("Could not locate the project root.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
project_root


In [ ]:
from configs import compose_experiment_config
from predictors.lstm_forecaster import LSTMForecaster
from predictors.training import (
    collect_available_lstm_artifacts,
    load_signal_matrix_from_source,
    plot_signal_training_report,
    resolve_signal_csv_source,
    train_signal_lstm,
)
from scripts.utils.forecast_diagnostics import run_sfh14_load_repair_report
from scripts.utils.torch_runtime import configure_torch_runtime, describe_device


## 2. Experiment Controls

Edit this block to switch between fresh training and isolated artifact reuse.


In [ ]:
runtime_mode = "performance"
seed = 0
device_request = None
require_cuda = False

focus_profile = "SFH14"
control_profiles = ["SFH12", "SFH16"]
agent_profiles = ["SFH12", "SFH14", "SFH16"]
retrain = True

load_time_feature_mode = "hour_week_year"
history_window = 96 * 7
future_horizon = 24
artifact_root = project_root / "artifacts" / "forecast" / "experiments" / "sfh14" / "lstm"
analysis_output_dir = project_root / "artifacts" / "forecast" / "experiments" / "sfh14" / "analysis" / "sfh14_load"

load_training_overrides = {
    "hidden_size": 96,
    "num_layers": 2,
    "dropout": 0.10,
    "batch_size": 1024,
    "epochs": 10,
    "lr": 1e-3,
}


In [ ]:
cfg = compose_experiment_config(
    forecast_type="lstm",
    data_dir=project_root / "data",
    runtime_mode=runtime_mode,
    seed=seed,
    require_cuda=require_cuda,
)
cfg.data.agent_profiles = agent_profiles
cfg.data.train_year = 2019
cfg.data.test_year = 2020
cfg.data.load_components = ["household", "heatpump"]
cfg.obs.sequence_features = ["load"]
cfg.forecast.target_signals = ["load"]
cfg.env.future_horizon = int(future_horizon)
cfg.forecast.history_window = int(history_window)
cfg.forecast.load_model_mode = "per_agent"
cfg.forecast.load_time_feature_mode = load_time_feature_mode
cfg.forecast.load_hybrid_mode = "baseline_blend"
cfg.forecast.load_baseline_mode = "last_value"
cfg.forecast.auto_train_missing = False
cfg.forecast.lstm_artifact_root = Path(artifact_root)

runtime_state = configure_torch_runtime(
    cfg,
    device=device_request,
    seed=seed,
    require_cuda=require_cuda,
)

experiment_summary = {
    "device": str(runtime_state.device),
    "device_info": describe_device(runtime_state),
    "focus_profile": focus_profile,
    "control_profiles": control_profiles,
    "agent_profiles": agent_profiles,
    "retrain": retrain,
    "history_window": int(cfg.forecast.history_window),
    "future_horizon": int(cfg.env.future_horizon),
    "load_time_feature_mode": cfg.forecast.load_time_feature_mode,
    "artifact_root": str(cfg.forecast.lstm_artifact_root),
    "analysis_output_dir": str(analysis_output_dir),
    "load_training_overrides": load_training_overrides,
}
pprint(experiment_summary)


## 3. Baseline Training or Artifact Reuse

Use `retrain = True` for a fresh baseline. Use `retrain = False` to reuse only the isolated SFH14 experiment artifacts.


In [ ]:
artifact_overrides = {"load": dict(load_training_overrides)}


def _normalize_artifact_paths(artifact_paths):
    if isinstance(artifact_paths, list):
        normalized = []
        for item in artifact_paths:
            if isinstance(item, dict):
                normalized.append((item["model_path"], item["meta_path"], item["scaler_path"]))
            else:
                normalized.append(tuple(item))
        return normalized
    if isinstance(artifact_paths, dict):
        return [(artifact_paths["model_path"], artifact_paths["meta_path"], artifact_paths["scaler_path"])]
    if isinstance(artifact_paths, tuple):
        return [tuple(artifact_paths)]
    raise TypeError(f"Unsupported artifact payload: {type(artifact_paths)!r}")


def train_and_report_load():
    print("=" * 88)
    print("[load] training starts")
    pprint(load_training_overrides)
    result = train_signal_lstm(
        cfg,
        "load",
        device=runtime_state,
        overrides=load_training_overrides,
        show_progress=True,
    )
    figure = plot_signal_training_report(result)
    plt.show()
    plt.close(figure)
    print(f"[load] best_val_loss = {result['training']['best_val_loss']:.6f}")
    if result.get("agent_results"):
        for agent_result in result["agent_results"]:
            print(
                f"[load] agent={agent_result['agent_profile']} "
                f"best_val_loss={agent_result['training']['best_val_loss']:.6f} "
                f"blend_weight={agent_result['hybrid']['blend_weight']} "
                f"artifact_paths={agent_result['artifact_paths']}"
            )
    return result


def reuse_isolated_load_result():
    artifact_map = collect_available_lstm_artifacts(cfg, overrides_by_signal=artifact_overrides)
    artifact_paths = artifact_map.get("load")
    if artifact_paths is None:
        raise FileNotFoundError(
            f"No isolated load artifacts found under {cfg.forecast.lstm_artifact_root}. Run once with retrain=True."
        )
    load_source = resolve_signal_csv_source(Path(cfg.data.data_dir), "load", cfg=cfg)
    if load_source is None:
        raise FileNotFoundError("Could not resolve the load source from the current config.")
    reused = {
        "artifact_paths": artifact_paths,
        "source": load_source,
        "settings": {"history_window": int(cfg.forecast.history_window)},
    }
    if isinstance(artifact_paths, list):
        reused["agent_results"] = [
            {"agent_profile": profile, "artifact_paths": bundle}
            for profile, bundle in zip(cfg.data.agent_profiles, _normalize_artifact_paths(artifact_paths))
        ]
    return reused


if retrain:
    load_result = train_and_report_load()
else:
    load_result = reuse_isolated_load_result()
    print(f"[load] reusing isolated artifacts from {cfg.forecast.lstm_artifact_root}")
    pprint(load_result["artifact_paths"])

load_result["artifact_paths"]


## 4. Full Test-Set View with Ideal Warmup

This view keeps the same ideal warmup assumption as the generic notebook, but focuses only on the `load` signal.


In [ ]:
def _build_load_forecaster_from_result():
    normalized = _normalize_artifact_paths(load_result["artifact_paths"])
    if len(normalized) > 1:
        return LSTMForecaster.from_signal_artifacts({"load": normalized}, device=runtime_state.device)
    model_path, meta_path, scaler_path = normalized[0]
    return LSTMForecaster.from_artifacts(
        model_path=model_path,
        meta_path=meta_path,
        scaler_path=scaler_path,
        device=runtime_state.device,
        signal_name="load",
    )


def _rolling_load_predictions_with_ideal_warmup():
    load_source = load_result.get("source")
    if load_source is None:
        load_source = resolve_signal_csv_source(Path(cfg.data.data_dir), "load", cfg=cfg)
    if load_source is None:
        raise FileNotFoundError("Could not resolve the load source from the current config.")

    test_frame, test_values, value_columns = load_signal_matrix_from_source(load_source, "load", split="test")
    forecaster = _build_load_forecaster_from_result()
    history_window = int(load_result["settings"]["history_window"])

    values = np.asarray(test_values, dtype=np.float32)
    if values.ndim == 1:
        values = values.reshape(-1, 1)

    working_frame = test_frame.copy()
    if "segment_id" not in working_frame.columns:
        working_frame["segment_id"] = 0
    working_frame["segment_id"] = working_frame["segment_id"].fillna(0).astype(int)

    segment_views = []
    for segment_id in sorted(working_frame["segment_id"].unique()):
        segment_mask = working_frame["segment_id"] == segment_id
        segment_frame = working_frame.loc[segment_mask].reset_index(drop=True)
        segment_values = values[segment_mask.to_numpy()]

        if segment_values.shape[0] <= history_window:
            print(
                f"[skip] load, segment_id={segment_id}: rows={segment_values.shape[0]} <= history_window={history_window}"
            )
            continue

        timestamps = []
        target_rows = []
        prediction_rows = []
        for end_idx in range(history_window, segment_values.shape[0]):
            history = segment_values[end_idx - history_window:end_idx]
            history_timestamps = (
                segment_frame.iloc[end_idx - history_window:end_idx]["timestamp"].tolist()
                if "timestamp" in segment_frame.columns
                else None
            )
            rollout = forecaster.predict(
                history,
                horizon=2,
                signal_name="load",
                history_timestamps=history_timestamps,
            )
            rollout = np.asarray(rollout, dtype=np.float32)
            if rollout.ndim == 1:
                prediction_rows.append(np.array([rollout[1]], dtype=np.float32))
            else:
                prediction_rows.append(rollout[:, 1].astype(np.float32))
            target_rows.append(segment_values[end_idx].astype(np.float32))
            timestamps.append(segment_frame.iloc[end_idx]["timestamp"] if "timestamp" in segment_frame.columns else end_idx)

        segment_views.append(
            {
                "segment_id": int(segment_id),
                "timestamps": np.asarray(timestamps),
                "target": np.stack(target_rows, axis=0),
                "prediction": np.stack(prediction_rows, axis=0),
            }
        )

    if not segment_views:
        raise ValueError(f"load has no test segment long enough for history_window={history_window}.")

    profile_names = [column.split("_", maxsplit=1)[-1] for column in value_columns]
    return {
        "segments": segment_views,
        "value_columns": value_columns,
        "profile_names": profile_names,
        "history_window": history_window,
    }


full_test_view = _rolling_load_predictions_with_ideal_warmup()

figure, axes = plt.subplots(len(full_test_view["value_columns"]), 1, figsize=(16, 10), sharex=False)
axes = np.atleast_1d(axes)
for profile_idx, axis in enumerate(axes):
    profile_name = full_test_view["profile_names"][profile_idx]
    for segment in full_test_view["segments"]:
        axis.plot(
            segment["timestamps"],
            segment["target"][:, profile_idx],
            linewidth=1.5,
            label=f"Ground truth (segment {segment['segment_id']})",
        )
        axis.plot(
            segment["timestamps"],
            segment["prediction"][:, profile_idx],
            linewidth=1.2,
            linestyle="--",
            label=f"Prediction (segment {segment['segment_id']})",
        )
    axis.set_title(
        f"load - {profile_name} rolling one-step forecast by segment (ideal warmup={full_test_view['history_window']})"
    )
    axis.set_ylabel(profile_name)
    axis.grid(True, alpha=0.3)
    axis.legend(loc="upper right", ncol=2)

axes[-1].set_xlabel("timestamp")
figure.tight_layout()
plt.show()

full_test_view


## 5. Diagnostics and Offline Repair Experiments

This section writes the isolated report tables, then surfaces both the step-1 diagnostics and the new 24-step rollout diagnostics side by side.


In [ ]:
load_analysis_report = run_sfh14_load_repair_report(
    cfg,
    runtime_state=runtime_state,
    load_result=load_result,
    load_overrides=load_training_overrides,
    focus_profile=focus_profile,
    output_dir=analysis_output_dir,
    show_progress=False,
)

for table_name in (
    "step1_metrics",
    "sfh14_monthly_metrics",
    "horizon_metrics",
    "sfh14_month_horizon_metrics",
    "component_drift",
    "experiment_summary",
    "rollout_experiment_summary",
):
    print(f"--- {table_name} ---")
    display(load_analysis_report[table_name])

extra_tables = {
    "e2_candidate_metrics": load_analysis_report["experiments"]["e2"]["candidate_metrics"],
    "e2_block_metrics": load_analysis_report["experiments"]["e2"]["block_metrics"],
    "e3_total_metrics": load_analysis_report["experiments"]["e3"]["total_metrics"],
}
for table_name, table in extra_tables.items():
    print(f"--- {table_name} ---")
    if getattr(table, "empty", False):
        print("empty")
        continue
    display(table)

print("written artifacts:")
pprint({key: str(path) for key, path in load_analysis_report.get("written_paths", {}).items()})


## 6. Recommendation

Use this output to decide whether the issue is isolated enough to fold back into the generic notebook or whether the next step needs weather features.


In [ ]:
recommendation = load_analysis_report["recommendation"]
print("recommendation:")
pprint(recommendation)

if recommendation["preferred_experiment"] is None:
    print(
        "Conclusion: no artifact-only repair cleared the acceptance targets. Add weather features before integrating changes back into the generic notebook."
    )
else:
    print(
        f"Conclusion: continue with {recommendation['preferred_experiment']} / {recommendation['preferred_variant']} before integrating back into forecast_lstm.ipynb."
    )
